In [1]:
# Scraped from: Wikibooks (https://en.wikibooks.org/wiki/Cookbook:Cuisines)
import requests
from bs4 import BeautifulSoup
import json
import time
from urllib.parse import unquote

def scrape_dynamic_structured_data(links, headers):
    collected_data = []
    ignore_headings = ['contents', 'see also', 'references', 'external links', 'navigation menu']

    print("Initializing dynamic extraction from Wikibooks...")
    
    for index, url in enumerate(links, 1):
        print(f"[{index:02d}/{len(links)}] Parsing: {unquote(url)}")
        try:
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.content, "html.parser")
            all_tags = soup.find_all(True)
            
            recipe_sections = {}
            active_main_section = None
            active_level = 0
            
            for i, tag in enumerate(all_tags):
                if tag.name in ['h2', 'h3']:
                    raw_text = tag.get_text(strip=True).replace('[edit | edit source]', '').replace('[edit]', '').strip()
                    lower_text = raw_text.lower()
                    
                    if not raw_text or lower_text in ignore_headings:
                        active_main_section = None 
                        continue
                        
                    current_level = int(tag.name[1])
                    
                    if current_level == 2 or (current_level == 3 and not active_main_section):
                        active_main_section = raw_text
                        active_level = current_level
                        if active_main_section not in recipe_sections:
                            recipe_sections[active_main_section] = []
                            
                    elif current_level > active_level and active_main_section:
                        recipe_sections[active_main_section].append(f"\n[{raw_text}]")

                elif active_main_section:
                    if tag.name == 'li':
                        text = tag.get_text(separator=' ', strip=True)
                        if text: recipe_sections[active_main_section].append("- " + text)
                    elif tag.name == 'p':
                        text = tag.get_text(separator=' ', strip=True)
                        if text: recipe_sections[active_main_section].append(text)

            final_sections = {}
            for key, lines in recipe_sections.items():
                unique_list = []
                for line in lines:
                    if line not in unique_list: unique_list.append(line)
                
                cleaned_text = "\n".join(unique_list).strip()
                if cleaned_text and any(char.isalnum() for char in cleaned_text.replace('[', '').replace(']', '')):
                    final_sections[key] = cleaned_text

            if not final_sections:
                print("      [Warning] No valid content sections found.")
            else:
                title_tag = soup.find("h1", id="firstHeading")
                title = title_tag.get_text(strip=True) if title_tag else "Untitled"
                
                collected_data.append({
                    "id": f"med_recipe_{index:02d}",
                    "url": url,
                    "title": title,
                    "content_sections": final_sections
                })
                found_keys = list(final_sections.keys())
                print(f"      [Success] Extracted sections: {', '.join(found_keys)}")
            
            time.sleep(1.5)
        except Exception as e:
            print(f"      [Error] Failed to process URL: {e}")

    output_filename = "Wikibooks-scraped_data.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(collected_data, f, ensure_ascii=False, indent=4)
    
    print(f"\nExtraction complete. Data saved to: {output_filename}")


if __name__ == "__main__":
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    recipe_links = [
        "https://en.wikibooks.org/wiki/Cookbook:Arroz_Negro_(Valencian_Squid_Rice)",
        "https://en.wikibooks.org/wiki/Cookbook:Tzatziki",
        "https://en.wikibooks.org/wiki/Cookbook:Pudding_with_Pistachios_and_Rose_Water_(Muhallebi)",
        "https://en.wikibooks.org/wiki/Cookbook:Menemen_(Eggs_with_Onion,_Green_Pepper,_and_Tomato)",
        "https://en.wikibooks.org/wiki/Cookbook:Turkish_Delight",
        "https://en.wikibooks.org/wiki/Cookbook:Ayran_(Turkish_Yogurt_Drink)",
        "https://en.wikibooks.org/wiki/Cookbook:Egg_with_Cabbage_and_Turkish_Sausage",
        "https://en.wikibooks.org/wiki/Cookbook:Greek_Chicken_Wrap",
        "https://en.wikibooks.org/wiki/Cookbook:Paella_Valenciana",
        "https://en.wikibooks.org/wiki/Cookbook:Kebab",
        "https://en.wikibooks.org/wiki/Cookbook:Rosemary_Garlic_Fish",
        "https://en.wikibooks.org/wiki/Cookbook:Pan-Fried_White_Brined_Cheese_(Saganaki)",
        "https://en.wikibooks.org/wiki/Cookbook:Baklava_with_Pistachio_Nuts",
        "https://en.wikibooks.org/wiki/Cookbook:Börek_(Turkish_Filled_Pastries)",
        "https://en.wikibooks.org/wiki/Cookbook:Hummus_(Greek)",
        "https://en.wikibooks.org/wiki/Cookbook:Spaghetti_alla_Puttanesca",
        "https://en.wikibooks.org/wiki/Cookbook:Mediterranean_Green_Chicken",
        "https://en.wikibooks.org/wiki/Cookbook:Greek_Yogurt_Sauce",
        "https://en.wikibooks.org/wiki/Cookbook:Eggplant_and_Tahini_(Moutabbal)",
        "https://en.wikibooks.org/wiki/Cookbook:Galaktoboureko_(Greek_Semolina_Custard_Pastry)",
        "https://en.wikibooks.org/wiki/Cookbook:Baklava_I",
        "https://en.wikibooks.org/wiki/Cookbook:Beef_and_Vegetable_Kabobs",
        "https://en.wikibooks.org/wiki/Cookbook:Yuvarlak_(Greek_Meatballs)",
        "https://en.wikibooks.org/wiki/Cookbook:Greek-Style_Grilled_Chicken",
        "https://en.wikibooks.org/wiki/Cookbook:Loukoumas_(Greek_Donuts_with_Honey)_II",
        "https://en.wikibooks.org/wiki/Cookbook:Turkish_Dolma_Pilav",
        "https://en.wikibooks.org/wiki/Cookbook:Paella_de_Marisco",
        "https://en.wikibooks.org/wiki/Cookbook:Frappé_Coffee",
        "https://en.wikibooks.org/wiki/Cookbook:Cheese_Filling_for_Peynirli_Börek",
        "https://en.wikibooks.org/wiki/Cookbook:Shakshuka_I",
        "https://en.wikibooks.org/wiki/Cookbook:Greek_Moussaka",
        "https://en.wikibooks.org/wiki/Cookbook:Baba_Ganoush",
        "https://en.wikibooks.org/wiki/Cookbook:Paella_Roja",
        "https://en.wikibooks.org/wiki/Cookbook:Mediterranean_Grilled_Tuna",
        "https://en.wikibooks.org/wiki/Cookbook:Gözleme_(Turkish_Flatbread)",
        "https://en.wikibooks.org/wiki/Cookbook:Pita",
        "https://en.wikibooks.org/wiki/Cookbook:Pasta_Casserole_(Pastitsio)",
        "https://en.wikibooks.org/wiki/Cookbook:Maltese_Rabbit_Stew_(Stuffat_tal-Fenek)",
        "https://en.wikibooks.org/wiki/Cookbook:Stuffed_Grape_Leaves",
        "https://en.wikibooks.org/wiki/Cookbook:Baked_Chicken_with_Onions,_Sumac_and_Allspice_(Musakhan)",
        "https://en.wikibooks.org/wiki/Cookbook:Tyropita_(Greek_Cheese_Pie)",
        "https://en.wikibooks.org/wiki/Cookbook:Spanakopita",
        "https://en.wikibooks.org/wiki/Cookbook:Döner_Kebab",
        "https://en.wikibooks.org/wiki/Cookbook:Mediterranean_Beef_Stew",
        "https://en.wikibooks.org/wiki/Cookbook:Loukoumas_(Greek_Donuts_with_Honey)_I",
        "https://en.wikibooks.org/wiki/Cookbook:Arjoli_(Maltese_Herbed_Bread_Dip)",
        "https://en.wikibooks.org/wiki/Cookbook:Cypriot_Salad",
        "https://en.wikibooks.org/wiki/Cookbook:Valencian-Inspired_Paella",
        "https://en.wikibooks.org/wiki/Cookbook:Baklava_II",
        "https://en.wikibooks.org/wiki/Cookbook:Tarhana_(Turkish_Yogurt_Soup)",
        "https://en.wikibooks.org/wiki/Cookbook:Kawlata_(Maltese_Vegetable_and_Pork_Soup)"
    ]

    scrape_dynamic_structured_data(recipe_links, headers)

Initializing dynamic extraction from Wikibooks...
[01/51] Parsing: https://en.wikibooks.org/wiki/Cookbook:Arroz_Negro_(Valencian_Squid_Rice)
      [Success] Extracted sections: Ingredients, Equipment, Procedure, Notes, tips, and variations
[02/51] Parsing: https://en.wikibooks.org/wiki/Cookbook:Tzatziki
      [Success] Extracted sections: Ingredients, Procedure
[03/51] Parsing: https://en.wikibooks.org/wiki/Cookbook:Pudding_with_Pistachios_and_Rose_Water_(Muhallebi)
      [Success] Extracted sections: Ingredients, Procedure, Notes, tips, and variations
[04/51] Parsing: https://en.wikibooks.org/wiki/Cookbook:Menemen_(Eggs_with_Onion,_Green_Pepper,_and_Tomato)
      [Success] Extracted sections: Ingredients, Procedure, Notes
[05/51] Parsing: https://en.wikibooks.org/wiki/Cookbook:Turkish_Delight
      [Success] Extracted sections: Ingredients, Procedure, Notes, tips, and variations, Warnings
[06/51] Parsing: https://en.wikibooks.org/wiki/Cookbook:Ayran_(Turkish_Yogurt_Drink)
      [Succe

In [7]:
# Scraped from: Around the World in 80 Cuisines (https://aroundtheworldin80cuisinesblog.wordpress.com/)
import requests
from bs4 import BeautifulSoup
import json
import time

def scrape_wp_blog(urls, headers):
    collected_data = []
    
    print("Initializing dynamic extraction from 80 Cuisines blog...")
    
    for index, url in enumerate(urls, 1):
        print(f"[{index:02d}/{len(urls)}] Parsing: {url}")
        try:
            response = requests.get(url, headers=headers)
            
            if response.status_code != 200:
                print(f"      [Warning] HTTP Error {response.status_code}.")
                time.sleep(2)
                continue
                
            soup = BeautifulSoup(response.content, "html.parser")
            
            title_tag = soup.find("h1")
            title = title_tag.get_text(strip=True) if title_tag else f"Cuisine_Region_{index}"
            
            content_div = soup.find("div", class_="entry-content")
            
            if not content_div:
                paragraphs = soup.find_all("p")
            else:
                paragraphs = content_div.find_all("p")
                
            text_lines = []
            for p in paragraphs:
                text = p.get_text(separator=' ', strip=True)
                if text:
                    text_lines.append(text)
                    
            full_text = "\n\n".join(text_lines)
            
            if full_text:
                collected_data.append({
                    "id": f"80_cuisines_{index:02d}",
                    "source": "Around the world in 80 cuisines",
                    "url": url,
                    "title": title,
                    "content_sections": {
                        "Blog Content": full_text
                    }
                })
                print("      [Success] Extracted blog content.")
            else:
                print("      [Warning] No valid text content extracted.")
                
            time.sleep(1.5) 
            
        except Exception as e:
            print(f"      [Error] Failed to process URL: {e}")
            
    output_file = "80Cuisines-scraped_data.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(collected_data, f, ensure_ascii=False, indent=4)
        
    print(f"\nExtraction complete. Data saved to: {output_file}")

if __name__ == "__main__":
    custom_headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    target_urls = [
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/01-southern-france-and-monaco/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/04-turkey-and-cyprus/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/13-north-africa/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/27-former-yugoslavia/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/33-greece/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/37-southern-italy/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/47-central-italy/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/53-spain/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/69-northern-italy/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/08-south-asia-minor/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/15-portugal/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/58-the-fertile-crescent/",
        "https://aroundtheworldin80cuisinesblog.wordpress.com/category/66-germany/"
    ]
    
    scrape_wp_blog(target_urls, custom_headers)

Initializing dynamic extraction from 80 Cuisines blog...
[01/13] Parsing: https://aroundtheworldin80cuisinesblog.wordpress.com/category/01-southern-france-and-monaco/
      [Success] Extracted blog content.
[02/13] Parsing: https://aroundtheworldin80cuisinesblog.wordpress.com/category/04-turkey-and-cyprus/
      [Success] Extracted blog content.
[03/13] Parsing: https://aroundtheworldin80cuisinesblog.wordpress.com/category/13-north-africa/
      [Success] Extracted blog content.
[04/13] Parsing: https://aroundtheworldin80cuisinesblog.wordpress.com/category/27-former-yugoslavia/
      [Success] Extracted blog content.
[05/13] Parsing: https://aroundtheworldin80cuisinesblog.wordpress.com/category/33-greece/
      [Success] Extracted blog content.
[06/13] Parsing: https://aroundtheworldin80cuisinesblog.wordpress.com/category/37-southern-italy/
      [Success] Extracted blog content.
[07/13] Parsing: https://aroundtheworldin80cuisinesblog.wordpress.com/category/47-central-italy/
      [Suc

In [9]:
# Standardizes JSON key order and injects source metadata for Wikibooks recipes.
import json

def align_exact_keys_order(input_file, output_file):
    print("Standardizing JSON key order...")
    
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            recipes = json.load(f)
    except FileNotFoundError:
        print(f"      [Error] File not found: {input_file}")
        return
        
    ordered_data = []
    
    for recipe in recipes:
        ordered_recipe = {}
        ordered_recipe["id"] = recipe.get("id", "")
        ordered_recipe["source"] = "Wikibooks-cookbook"
        ordered_recipe["url"] = recipe.get("url", "")
        ordered_recipe["title"] = recipe.get("title", "")
        ordered_recipe["content_sections"] = recipe.get("content_sections", {})
            
        ordered_data.append(ordered_recipe)
        
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(ordered_data, f, ensure_ascii=False, indent=4)
        
    print(f"      [Success] Processed and reordered {len(ordered_data)} recipes.")
    print(f"Data saved to: {output_file}\n")


if __name__ == "__main__":
    input_file = "Wikibooks-scraped_data.json"
    output_file = "Wikibooks-cleaned_data.json"
    
    align_exact_keys_order(input_file, output_file)

Standardizing JSON key order...
      [Success] Processed and reordered 50 recipes.
Data saved to: Wikibooks-cleaned_data.json



In [1]:
# Scraped from: Wikipedia (https://en.wikipedia.org/wiki/List_of_cuisines)
from __future__ import annotations

import argparse
import hashlib
import json
import re
import time
from pathlib import Path
from typing import Dict, List, Optional

import requests
from bs4 import BeautifulSoup

DEFAULT_OUTPUT = Path("Wikipedia-scraped_data.json")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; LogicCatFetcher/1.1; +https://example.com)"
}

SOURCE_CONFIG = {
    "default": {"retries": 3, "backoff": 1.5, "sleep": 0.5},
    "WordPress": {"retries": 5, "backoff": 2.5, "sleep": 2.0},
}

URL_CATALOG: Dict[str, List[Dict[str, str]]] = {
    "Wikipedia": [
        {"title": "Mediterranean cuisine", "url": "https://en.wikipedia.org/wiki/Mediterranean_cuisine"},
        {"title": "Italian cuisine", "url": "https://en.wikipedia.org/wiki/Italian_cuisine"},
        {"title": "Greek cuisine", "url": "https://en.wikipedia.org/wiki/Greek_cuisine"},
        {"title": "Spanish cuisine", "url": "https://en.wikipedia.org/wiki/Spanish_cuisine"},
        {"title": "Lebanese cuisine", "url": "https://en.wikipedia.org/wiki/Lebanese_cuisine"},
        {"title": "Moroccan cuisine", "url": "https://en.wikipedia.org/wiki/Moroccan_cuisine"},
        {"title": "Turkish cuisine", "url": "https://en.wikipedia.org/wiki/Turkish_cuisine"},
        {"title": "French cuisine", "url": "https://en.wikipedia.org/wiki/French_cuisine"},
        {"title": "Cypriot cuisine", "url": "https://en.wikipedia.org/wiki/Cypriot_cuisine"},
        {"title": "Israeli cuisine", "url": "https://en.wikipedia.org/wiki/Israeli_cuisine"},
        {"title": "Tunisian cuisine", "url": "https://en.wikipedia.org/wiki/Tunisian_cuisine"},
        {"title": "Levantine cuisine", "url": "https://en.wikipedia.org/wiki/Levantine_cuisine"},
        {"title": "Sicilian cuisine", "url": "https://en.wikipedia.org/wiki/Sicilian_cuisine"},
        {"title": "Catalan cuisine", "url": "https://en.wikipedia.org/wiki/Catalan_cuisine"},
        {"title": "Provençal cuisine", "url": "https://en.wikipedia.org/wiki/Proven%C3%A7al_cuisine"},
        {"title": "Egyptian cuisine", "url": "https://en.wikipedia.org/wiki/Egyptian_cuisine"},
        {"title": "Algerian cuisine", "url": "https://en.wikipedia.org/wiki/Algerian_cuisine"},
        {"title": "Syrian cuisine", "url": "https://en.wikipedia.org/wiki/Syrian_cuisine"},
        {"title": "Palestinian cuisine", "url": "https://en.wikipedia.org/wiki/Palestinian_cuisine"},
        {"title": "Portuguese cuisine", "url": "https://en.wikipedia.org/wiki/Portuguese_cuisine"},
    ]
}

FOOTER_PATTERNS = [
    re.compile(r"This page was last edited.*", re.IGNORECASE),
    re.compile(r"Text is available under.*", re.IGNORECASE),
    re.compile(r"Privacy policy.*", re.IGNORECASE),
    re.compile(r"Terms of Use.*", re.IGNORECASE),
]

CITATION_PATTERN = re.compile(r"\[\s*\d+\s*\]")
MULTISPACE_PATTERN = re.compile(r"\s{2,}")
NEWLINE_LITERAL_PATTERN = re.compile(r"\\n+")
NON_ASCII_PATTERN = re.compile(r"[\u4e00-\u9fff]+")


def flatten_targets(limit: Optional[int] = None) -> List[Dict[str, str]]:
    targets: List[Dict[str, str]] = []
    for source, entries in URL_CATALOG.items():
        for idx, entry in enumerate(entries, start=1):
            targets.append({
                "id": f"{source.lower()}_{idx:03d}",
                "source": source,
                "title": entry["title"],
                "url": entry["url"],
            })
    if limit:
        targets = targets[:limit]
    return targets


def get_source_config(source: str) -> Dict[str, float]:
    cfg = SOURCE_CONFIG.get("default", {}).copy()
    cfg.update(SOURCE_CONFIG.get(source, {}))
    return cfg


def cache_filename(cache_dir: Path, url: str) -> Path:
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()
    return cache_dir / f"{digest}.html"


def fetch_html(url: str, source: str, cache_dir: Optional[Path], cfg: Dict[str, float]) -> Optional[str]:
    if cache_dir:
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_file = cache_filename(cache_dir, url)
        if cache_file.exists():
            return cache_file.read_text(encoding="utf-8", errors="ignore")
    else:
        cache_file = None

    retries = int(cfg.get("retries", 3))
    backoff = float(cfg.get("backoff", 1.5))

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            html = resp.text
            if cache_file:
                cache_file.write_text(html, encoding="utf-8", errors="ignore")
            return html
        except Exception as exc:
            if attempt == retries:
                print(f"[!] Failed {url}: {exc}")
                return None
            sleep_for = backoff * attempt
            print(f"[~] Retry {attempt}/{retries} for {url} in {sleep_for:.1f}s ...")
            time.sleep(sleep_for)
    return None


def clean_heading(raw: str) -> str:
    cleaned = raw.replace("[edit]", "").strip()
    return cleaned or "Overview"


def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = CITATION_PATTERN.sub("", text)
    text = NEWLINE_LITERAL_PATTERN.sub(" ", text)
    for pattern in FOOTER_PATTERNS:
        text = pattern.sub("", text)
    text = MULTISPACE_PATTERN.sub(" ", text)
    return text.strip()


def extract_sections(html: str) -> List[Dict[str, str]]:
    soup = BeautifulSoup(html, "html.parser")
    content_root = (
        soup.find("div", id="mw-content-text")
        or soup.find("div", class_="entry-content")
        or soup.find("article")
        or soup.body
        or soup
    )

    sections: List[Dict[str, str]] = []
    current = {"heading": "Overview", "level": 1, "content": []}

    for elem in content_root.descendants:
        if elem.name in ("h1", "h2", "h3", "h4"):
            heading = clean_heading(elem.get_text(" ", strip=True))
            level = int(elem.name[-1]) if elem.name[-1].isdigit() else 1
            if current["content"]:
                sections.append({
                    "heading": current["heading"],
                    "level": current["level"],
                    "content": clean_text("\n".join(current["content"])),
                })
            current = {"heading": heading, "level": level, "content": []}
        elif elem.name in ("p", "li"):
            text = elem.get_text(" ", strip=True)
            if text:
                current["content"].append(text)

    if current["content"]:
        sections.append({
            "heading": current["heading"],
            "level": current["level"],
            "content": clean_text("\n".join(current["content"])),
        })

    return [section for section in sections if section["content"]]


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--output", default=str(DEFAULT_OUTPUT), help="Path to save the structured corpus JSON")
    parser.add_argument("--sleep", type=float, default=0.5, help="Default sleep seconds between requests")

    parser.add_argument("--limit", type=int, default=20, help="Maximum number of URLs to fetch")
    parser.add_argument("--cache", type=str, default="html_cache", help="Directory to cache raw HTML (set '' to disable)")
    
    args, _ = parser.parse_known_args()

    SOURCE_CONFIG.setdefault("default", {})["sleep"] = args.sleep

    cache_dir = Path(args.cache) if args.cache else None
    targets = flatten_targets(limit=args.limit)

    records = []
    failed = []
    stats: Dict[str, int] = {}

    for idx, target in enumerate(targets, start=1):
        cfg = get_source_config(target["source"])
        print(f"[*] ({idx}/{len(targets)}) Fetching {target['title']} -> {target['url']}")
        html = fetch_html(target["url"], target["source"], cache_dir, cfg)
        if not html:
            failed.append(target)
            continue
        sections = extract_sections(html)
        if not sections:
            print(f"[!] No sections parsed for {target['url']}")
            failed.append(target)
            continue
        record = {
            "id": target["id"],
            "title": target["title"],
            "url": target["url"],
            "source": target["source"],
            "sections": sections,
        }
        records.append(record)
        stats[target["source"]] = stats.get(target["source"], 0) + 1
        time.sleep(cfg.get("sleep", args.sleep))

    output_path = Path(args.output)
    output_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[OK] Saved {len(records)} structured articles to {output_path.resolve()}")
    print(f"[STATS] per-source counts: {stats}")

    if failed:
        fail_path = output_path.with_suffix(output_path.suffix + ".failures.jsonl")
        with fail_path.open("w", encoding="utf-8") as fh:
            for item in failed:
                fh.write(json.dumps(item, ensure_ascii=False) + "\n")
        print(f"[!] {len(failed)} URLs failed. See {fail_path} for details.")


if __name__ == "__main__":
    main()

[*] (1/20) Fetching Mediterranean cuisine -> https://en.wikipedia.org/wiki/Mediterranean_cuisine
[*] (2/20) Fetching Italian cuisine -> https://en.wikipedia.org/wiki/Italian_cuisine
[*] (3/20) Fetching Greek cuisine -> https://en.wikipedia.org/wiki/Greek_cuisine
[*] (4/20) Fetching Spanish cuisine -> https://en.wikipedia.org/wiki/Spanish_cuisine
[*] (5/20) Fetching Lebanese cuisine -> https://en.wikipedia.org/wiki/Lebanese_cuisine
[*] (6/20) Fetching Moroccan cuisine -> https://en.wikipedia.org/wiki/Moroccan_cuisine
[*] (7/20) Fetching Turkish cuisine -> https://en.wikipedia.org/wiki/Turkish_cuisine
[*] (8/20) Fetching French cuisine -> https://en.wikipedia.org/wiki/French_cuisine
[*] (9/20) Fetching Cypriot cuisine -> https://en.wikipedia.org/wiki/Cypriot_cuisine
[*] (10/20) Fetching Israeli cuisine -> https://en.wikipedia.org/wiki/Israeli_cuisine
[*] (11/20) Fetching Tunisian cuisine -> https://en.wikipedia.org/wiki/Tunisian_cuisine
[*] (12/20) Fetching Levantine cuisine -> https://e

In [2]:
# Standardizes JSON key order and injects source metadata for Wikipedia recipes.
import json

with open('Wikipedia-scraped_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

wikipedia_data = [
    doc for doc in data 
    if doc.get("source") == "Wikipedia" or "wikipedia.org" in doc.get("url", "")
]
wikipedia_1_to_20 = wikipedia_data[:20]

blacklist = [
    "see also", 
    "references", 
    "bibliography", 
    "external links", 
    "further reading", 
    "notes"
]

formatted_data = []

for doc in wikipedia_1_to_20:
    new_doc = {
        "id": doc.get("id", ""),
        "source": doc.get("source", "Wikipedia"),
        "url": doc.get("url", ""),
        "title": doc.get("title", ""),
        "content_sections": {}  
    }
    
    if "sections" in doc and isinstance(doc["sections"], list):
        for sec in doc["sections"]:
            heading = sec.get("heading", "").strip()
            content = sec.get("content", "").strip()

            if not heading or heading.lower() in blacklist:
                continue

            new_doc["content_sections"][heading] = content
            
    formatted_data.append(new_doc)

with open('Wikipedia-cleaned_data.json', 'w', encoding='utf-8') as f:
    json.dump(formatted_data, f, indent=4, ensure_ascii=False)

print("Data structure reorganization completed! wikipedia_final_formatted.json has been generated.")

Data structure reorganization completed! wikipedia_final_formatted.json has been generated.


In [2]:
# Merges multiple JSON datasets into a single master corpus with a strictly enforced key structure.
import json

def merge_datasets_strictly(input_files, output_file):
    print("Initializing dataset merge and structure alignment...")
    
    master_corpus = []
    total_records = 0
    
    for file_path in input_files:
        print(f"      [Process] Reading: {file_path}")
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            for item in data:
                ordered_item = {
                    "id": item.get("id", ""),
                    "source": item.get("source", ""),
                    "url": item.get("url", ""),
                    "title": item.get("title", ""),
                    "content_sections": item.get("content_sections", {})
                }
                master_corpus.append(ordered_item)
                total_records += 1
                
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(master_corpus, f, ensure_ascii=False, indent=4)
        
    print(f"\n      [Success] Merge complete. {total_records} records saved to: {output_file}")

if __name__ == "__main__":
    files_to_merge = [
        "Wikibooks-cleaned_data.json",
        "80Cuisines-scraped_data.json",
        "Wikipedia-cleaned_data.json"
    ]
    
    final_output = "Background_Corpus.json"
    
    merge_datasets_strictly(files_to_merge, final_output)

Initializing dataset merge and structure alignment...
      [Process] Reading: Wikibooks-cleaned_data.json
      [Process] Reading: 80Cuisines-scraped_data.json
      [Process] Reading: Wikipedia-cleaned_data.json

      [Success] Merge complete. 83 records saved to: Background_Corpus.json
